# 03 · nn.Module: a pre-norm transformer block

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/pytorch/03_transformer_block.ipynb) &nbsp; 📖 Concept deck: [nn.Module and a transformer block](https://nilesh-salpe.github.io/visual-ai/pytorch/03-nn-module/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a causal self-attention module with `F.scaled_dot_product_attention`, an MLP, RMSNorm and residuals assembled into a pre-norm block, with parameter counting, `state_dict` and buffers.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)
print("PyTorch", torch.__version__)

## Task 1: Causal self-attention with SDPA

One `nn.Linear(C, 3C)` produces q, k, v; split into H heads; call `F.scaled_dot_product_attention(q, k, v, is_causal=True)`; merge heads; project with `nn.Linear(C, C)`. No biases (`bias=False`).
The test compares with an explicit masked-softmax implementation using your module's own weights.

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        self.H = H
        self.qkv = nn.Linear(C, 3 * C, bias=False)
        self.proj = nn.Linear(C, C, bias=False)
    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_CausalSelfAttention(cls):
    torch.manual_seed(0); B, T, C, H = 2, 6, 16, 4
    m = cls(C, H); x = torch.randn(B, T, C)
    y = m(x); assert y.shape == (B, T, C)
    q, k, v = m.qkv(x).split(C, dim=2)
    q, k, v = (t.view(B, T, H, C // H).transpose(1, 2) for t in (q, k, v))
    att = (q @ k.transpose(-2, -1)) / (C // H) ** 0.5
    att = att.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf")).softmax(-1)
    ref = m.proj((att @ v).transpose(1, 2).reshape(B, T, C))
    assert torch.allclose(y, ref, atol=1e-5), "must match masked softmax attention"
    y2 = m(torch.cat([x[:, :3], torch.randn(B, 3, C)], 1))
    assert torch.allclose(y[:, :3], y2[:, :3], atol=1e-5), "causal: future tokens must not affect earlier outputs"

check("CausalSelfAttention", _test_CausalSelfAttention, CausalSelfAttention)

## Task 2: A pre-norm block

`x = x + attn(RMSNorm(x)); x = x + mlp(RMSNorm(x))`, with `nn.RMSNorm(C)` and an MLP `Linear(C, 4C, bias=False) → GELU → Linear(4C, C, bias=False)`. Use the reference attention class `CausalSelfAttention_solution` if yours isn't done (`attn_cls` argument).
Parameter count: 3C² + C² (attention) + 8C² (MLP) + 2C (two norm gains) = 12C² + 2C.

In [ ]:
class Block(nn.Module):
    def __init__(self, C, H, attn_cls=None):
        super().__init__()
        attn_cls = attn_cls or CausalSelfAttention
        # TODO: self.n1, self.attn, self.n2, self.mlp
        raise NotImplementedError
    def forward(self, x):
        # TODO
        raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_Block(cls):
    C, H = 32, 4
    ref_attn = globals().get("CausalSelfAttention_solution") or CausalSelfAttention
    b = cls(C, H, ref_attn)
    n = sum(p.numel() for p in b.parameters())
    assert n == 12 * C * C + 2 * C, f"expected 12C² + 2C = {12 * C * C + 2 * C} parameters, got {n}"
    x = torch.randn(2, 5, C); y = b(x)
    assert y.shape == x.shape
    with torch.no_grad():
        for p in b.parameters(): p.zero_()
    assert torch.allclose(b(x), x), "with zero weights each residual branch outputs 0, so the block is the identity"

check("Block", _test_Block, Block)

## Task 3: Parameters vs buffers in state_dict

Register a non-trainable rotary-frequency table as a buffer. Write a tiny module `WithBuffers` that has one parameter `w` (shape (4,)), a persistent buffer `inv_freq` (shape (2,)) and a non-persistent buffer `cache` (shape (3,)).
Return `sorted(module.state_dict().keys())` and the names in `parameters()`. Persistent buffers are saved; non-persistent ones are not; neither is trained.

In [ ]:
class WithBuffers(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: nn.Parameter, register_buffer(..., persistent=False)
        raise NotImplementedError

def buffers_state():
    m = WithBuffers()
    return sorted(m.state_dict().keys()), [n for n, _ in m.named_parameters()]

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_buffers_state(f):
    keys, params = f()
    assert keys == ["inv_freq", "w"], f"state_dict keys: {keys}"
    assert params == ["w"], "buffers are not parameters"

check("buffers_state", _test_buffers_state, buffers_state)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Causal self-attention with SDPA
class CausalSelfAttention_solution(nn.Module):
    def __init__(self, C, H):
        super().__init__()
        self.H = H
        self.qkv = nn.Linear(C, 3 * C, bias=False)
        self.proj = nn.Linear(C, C, bias=False)
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.H, C // self.H).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

check("CausalSelfAttention (reference)", _test_CausalSelfAttention, CausalSelfAttention_solution)

In [ ]:
# @title Solution for Task 2: A pre-norm block
class Block_solution(nn.Module):
    def __init__(self, C, H, attn_cls=None):
        super().__init__()
        attn_cls = attn_cls or CausalSelfAttention_solution
        self.n1, self.attn = nn.RMSNorm(C), attn_cls(C, H)
        self.n2 = nn.RMSNorm(C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C, bias=False), nn.GELU(), nn.Linear(4 * C, C, bias=False))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))

check("Block (reference)", _test_Block, Block_solution)

In [ ]:
# @title Solution for Task 3: Parameters vs buffers in state_dict
class WithBuffers_solution(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(4))
        self.register_buffer("inv_freq", torch.ones(2))
        self.register_buffer("cache", torch.zeros(3), persistent=False)

def buffers_state_solution():
    m = WithBuffers_solution()
    return sorted(m.state_dict().keys()), [n for n, _ in m.named_parameters()]

check("buffers_state (reference)", _test_buffers_state, buffers_state_solution)

---
## Experiments
**Which SDPA backends exist?** On CPU, PyTorch uses its math or fused CPU kernels; on GPUs it can pick FlashAttention, memory-efficient attention or cuDNN. You can restrict the choice with `torch.nn.attention.sdpa_kernel`.

In [ ]:
from torch.nn.attention import SDPBackend, sdpa_kernel
print("backends:", [b for b in dir(SDPBackend) if b.isupper()])
q = k = v = torch.randn(1, 2, 8, 16)
with sdpa_kernel(SDPBackend.MATH):
    a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
b = F.scaled_dot_product_attention(q, k, v, is_causal=True)
print("math backend matches default:", torch.allclose(a, b, atol=1e-6))
blk = Block_solution(64, 8); print("block parameters:", sum(p.numel() for p in blk.parameters()), "= 12·64² + 2·64 =", 12 * 64 * 64 + 128)

---
## Takeaways
* `nn.Module` tracks parameters and sub-modules automatically; buffers hold non-trainable state (persistent ones go in `state_dict`).
* `F.scaled_dot_product_attention(…, is_causal=True)` replaces hand-written masked softmax and dispatches to fused kernels (FlashAttention on supported GPUs).
* A pre-norm block is two residual branches; with ≈12C² weights per block, parameter counts follow directly.